<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Create a Wide-Area Ethernet (Layer 2) Network: Automatic Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook does:** This notebook shows how to create an isolated **wide-area Layer 2 Ethernet** circuit connecting two nodes on **different** FABRIC sites, using **automatic** IP configuration. A WAN L2 network extends a single Ethernet segment across the FABRIC backbone, so both nodes share the same broadcast domain and subnet -- as if they were plugged into the same switch, even though they are at different physical locations.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Explain the difference between **local** and **wide-area** L2 networks in FABRIC
2. Create a WAN L2 network by placing interfaces on **different sites**
3. Use **automatic configuration** (`set_mode('auto')`) for IP assignment on WAN L2 networks
4. Verify cross-site L2 connectivity with `ping`

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Complete the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be comfortable creating basic slices (see [Hello, FABRIC](../../hello_fabric/hello_fabric.ipynb))
3. Understand local L2 networks (see [L2 Basic Auto](../create_l2network_basic/create_l2network_basic_auto.ipynb))

**Tip -- Auto vs. Manual:**
- **Auto** (this notebook): You specify a subnet, FABlib assigns IPs and configures interfaces
- **Manual** ([manual notebook](./create_l2network_wide_area_manual.ipynb)): You assign IPs yourself after the slice is active

</div>

## Background: Local vs. Wide-Area L2 Networks

FABRIC automatically determines whether to create a **local** or **wide-area** L2 network based on the location of the connected interfaces:

| Scenario | Network Type | What FABRIC Creates |
|----------|-------------|--------------------|
| All interfaces on the **same** site | Local L2 | A virtual switch at that site |
| Interfaces on **different** sites | Wide-Area L2 | A dedicated circuit across the backbone |

A WAN L2 network creates a **point-to-point Ethernet circuit** between two sites. Both nodes share the same subnet and broadcast domain, even though they may be thousands of kilometers apart.


### WAN L2 vs. FABnet (L3)

| Feature | WAN L2 | FABnet (L3) |
|---------|--------|-------------|
| **Addressing** | You choose the subnet | FABRIC assigns subnets |
| **Broadcast domain** | Shared across sites | Separate per site |
| **Routing** | None needed (same subnet) | Requires gateway routing |
| **Max sites** | 2 (point-to-point) | All FABRIC sites |
| **Use case** | Custom protocols, bridging | Standard IP routing |

### NIC Component Models

| Model | Speed | Type | Ports |
|-------|-------|------|-------|
| `NIC_Basic` | 100 Gbps | Mellanox ConnectX-6 SR-IOV VF | 1 |
| `NIC_ConnectX_5` | 25 Gbps | Dedicated Mellanox ConnectX-5 | 2 |
| `NIC_ConnectX_6` | 100 Gbps | Dedicated Mellanox ConnectX-6 | 2 |

## What We're Building

In this notebook we will create two nodes on different sites connected by a wide-area L2 Ethernet link (L2STS).

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import Python IP address management libraries
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Define Slice Parameters

We select two **different** random sites. Because the interfaces are on different sites, FABRIC will automatically create a wide-area L2 circuit.

In [ ]:
# Name for the slice
slice_name = 'MySlice'

# Pick two different random FABRIC sites
[site1,site2]  = fablib.get_random_sites(count=2)
print(f"Sites: {site1}, {site2}")

# Node and network names
node1_name = 'Node1'
node2_name = 'Node2'
network_name='net1'

## Step 3: Create the Slice with a WAN L2 Network

The key difference from a local L2 network is that we place nodes on **different sites**. FABRIC detects this and automatically creates a wide-area L2 circuit instead of a local switch.

We specify a **subnet** and set interface modes to `auto` so FABlib handles IP assignment.

<div class="fab-danger">

**Important:** WAN L2 networks in FABRIC support **exactly two sites** (point-to-point). If you need to connect more than two sites at Layer 2, you will need multiple WAN L2 networks or should consider using FABnet (L3) instead.

</div>

In [ ]:
# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# --- Create the L2 network with a user-defined subnet ---
# The subnet enables auto-configuration: FABlib assigns IPs from this range
net1 = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

# --- Node1 on site1 ---
node1 = slice.add_node(name=node1_name, site=site1)
# Add a NIC_Basic component and get its first (only) interface
iface1 = node1.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
# Set to 'auto' so FABlib assigns an IP from the subnet automatically
iface1.set_mode('auto')
# Connect the interface to the L2 network
net1.add_interface(iface1)

# --- Node2 on site2 (different site = WAN L2 circuit) ---
node2 = slice.add_node(name=node2_name, site=site2)
iface2 = node2.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
iface2.set_mode('auto')
net1.add_interface(iface2)

# Submit the slice request -- blocks until ready (~3-5 min)
slice.submit()

<div class="fab-success">

**What just happened?** FABRIC created two VMs on different sites, established a dedicated Layer 2 circuit between those sites through the FABRIC backbone, and during post-boot, FABlib assigned IP addresses from the `192.168.1.0/24` subnet and configured the network interfaces inside each VM. Both nodes are now on the same Ethernet segment.

</div>

## Step 4: Run the Experiment

With automatic configuration, the slice is ready immediately. We verify cross-site L2 connectivity by pinging Node2 from Node1.

<div class="fab-warning">

**Tip:** Since this is a WAN L2 link, the round-trip time will reflect the physical distance between the two sites (typically several milliseconds to tens of milliseconds, depending on geographic distance).

</div>

In [ ]:
# Retrieve the slice (useful if running this cell independently)
slice = fablib.get_slice(slice_name)

# Get the node objects
node1 = slice.get_node(name=node1_name)        
node2 = slice.get_node(name=node2_name)           

# Get Node2's automatically assigned IP address
node2_addr = node2.get_interface(network_name=network_name).get_ip_addr()

# Ping Node2 from Node1 over the WAN L2 circuit
stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

## Step 5: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources. WAN L2 circuits consume backbone capacity, so cleanup is especially important.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| Slice stuck in `Configuring` | Site may be busy or backbone link unavailable | Try different sites by re-running `get_random_sites()` |
| `ping` fails between nodes | Interfaces not configured or WAN circuit not established | Check `ip addr show` on both nodes; verify `set_mode('auto')` was called |
| Network created as local instead of WAN | Both nodes placed on the same site | Ensure `site1` and `site2` are different |
| `No resources available` | Site lacks NIC capacity | Choose different sites or try `NIC_ConnectX_6` |
| `submit()` times out | Network issue or high demand | Retry with `slice.submit(wait_timeout=600)` |
| High latency in ping | Sites are geographically distant | This is expected -- L2 WAN latency reflects physical distance |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.get_random_sites(count)` | Get a list of distinct random site names | [get_random_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_sites) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `fablib.get_slice(name)` | Retrieve an existing slice by name | [get_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_slice) |
| `slice.add_l2network(name, subnet)` | Add a Layer 2 network with a subnet | [add_l2network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l2network) |
| `slice.add_node(name, site)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |
| `node.add_component(model, name)` | Add a NIC or other component | [add_component](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_component) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `iface.set_mode('auto')` | Enable automatic IP configuration | [set_mode](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.set_mode) |
| `iface.get_ip_addr()` | Get the IP address assigned to an interface | [get_ip_addr](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.get_ip_addr) |
| `network.add_interface(iface)` | Connect an interface to a network | [add_interface](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.add_interface) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **L2 WAN Manual** | [create_l2network_wide_area_manual](./create_l2network_wide_area_manual.ipynb) | Assign IPs yourself for full control over WAN L2 |
| **L2 Local Auto** | [create_l2network_basic_auto](../create_l2network_basic/create_l2network_basic_auto.ipynb) | Create a local Ethernet on a single site |
| **FABnet IPv4 Auto** | [create_l3network_fabnet_ipv4_auto](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_auto.ipynb) | Layer 3 networking across all FABRIC sites |
| **FABnet IPv4 Full Auto** | [create_l3network_fabnet_ipv4_full_auto](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_full_auto.ipynb) | Simplest cross-site connectivity |
| **Hello FABRIC** | [hello_fabric](../../hello_fabric/hello_fabric.ipynb) | Start from the basics |